# **Feature Engineering: XGBoost for PJME Electricity Consumption Data**

In [26]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

In [27]:
df = pd.read_csv("../data/PJME_hourly.csv")
df = df.set_index('Datetime')
df.index = pd.to_datetime(df.index)
df = df.sort_index()

In [ ]:
from pandas.tseries.holiday import USFederalHolidayCalendar

def add_calendar_features(df):
  df = df.copy()
  df['hour']       = df.index.hour
  df['dayofweek']  = df.index.dayofweek
  df['month']      = df.index.month
  df['quarter']    = df.index.quarter
  df['year']       = df.index.year
  df['dayofyear']  = df.index.dayofyear
  df['dayofmonth'] = df.index.day
  df['weekofyear'] = df.index.isocalendar().week.astype(int)
  return df

def add_annual_lags(df):
  df = df.copy()
  target_map = df["PJME_MW"].to_dict()
  df["lag1"] = (df.index - pd.Timedelta('364 days')).map(target_map)
  df["lag2"] = (df.index - pd.Timedelta('728 days')).map(target_map)
  df["lag3"] = (df.index - pd.Timedelta('1092 days')).map(target_map)
  return df

def add_short_lags(df):
  df = df.copy()
  df['lag_24h']  = df['PJME_MW'].shift(24)
  df['lag_48h']  = df['PJME_MW'].shift(48)
  df['lag_168h'] = df['PJME_MW'].shift(168)
  return df

def add_rolling(df):
  df = df.copy()
  df['rolling_mean_24h']  = df['PJME_MW'].shift(1).rolling(24).mean()
  df['rolling_mean_168h'] = df['PJME_MW'].shift(1).rolling(168).mean()
  return df

def add_holidays(df):
  df = df.copy()
  cal      = USFederalHolidayCalendar()
  holidays = cal.holidays(start=df.index.min(), end=df.index.max())
  df['is_holiday'] = df.index.normalize().isin(holidays).astype(int)
  return df

# ClaudeAI for exogeneous variable temperature
def fetch_temperature(lat, lon, start, end):
  url = "https://archive-api.open-meteo.com/v1/archive"
  params = {
    "latitude":        lat,
    "longitude":       lon,
    "start_date":      start,
    "end_date":        end,
    "hourly":          "temperature_2m",
    "timezone":        "America/New_York",
  }
  r = requests.get(url, params=params)
  data = r.json()

  if "hourly" not in data:
    print(f"API error for ({lat}, {lon}): {data}")
    raise ValueError(f"No hourly data returned: {data}")

  temps = pd.Series(
    data["hourly"]["temperature_2m"],
    index=pd.to_datetime(data["hourly"]["time"]),
    name="temperature"
  )
  return temps


def fetch_avg_temperature(start, end):
  cities = {
    "Philadelphia": (39.9526, -75.1652),
    "Baltimore":    (39.2904, -76.6122),
    "Newark":       (40.7357, -74.1724),
    "Wilmington":   (39.7447, -75.5484),
  }
  series = []
  for city, (lat, lon) in cities.items():
    print(f"Fetching {city}...")
    series.append(fetch_temperature(lat, lon, start, end))

  avg = pd.concat(series, axis=1).mean(axis=1)
  avg.index = avg.index.tz_localize(None)  # strip timezone to match df index
  return avg


def add_temperature_features(df):
  df = df.copy()

  start = df.index.min().strftime("%Y-%m-%d")
  end   = df.index.max().strftime("%Y-%m-%d")

  print("Fetching temperature data from Open-Meteo...")
  avg_temp = fetch_avg_temperature(start, end)

  # Align to df index, forward fill any missing hours
  df["temperature"] = avg_temp.reindex(df.index, method="ffill")

  # Heating/cooling degrees (base 18.3°C)
  BASE = 18.3
  df["heating_degrees"] = (BASE - df["temperature"]).clip(lower=0)
  df["cooling_degrees"] = (df["temperature"] - BASE).clip(lower=0)

  # Rolling temperature (shift(1) to prevent leakage)
  df["rolling_temp_24h"]  = df["temperature"].shift(1).rolling(24).mean()
  df["rolling_temp_168h"] = df["temperature"].shift(1).rolling(168).mean()

  return df

def create_features(df):
  df = df.copy()
  df = add_calendar_features(df)
  df = add_annual_lags(df)
  df = add_short_lags(df)
  df = add_rolling(df)
  df = add_holidays(df)
  df = add_temperature_features(df)
  df['is_weekend']   = (df.index.dayofweek >= 5).astype(int)
  df['is_peak_hour'] = df.index.hour.isin(range(8, 21)).astype(int)
  df = df.dropna()
  return df

In [29]:
df = create_features(df)

print(f"No. rows after cleaning: {df.shape[0]}")
df.head(1)

Fetching temperature data from Open-Meteo...
Fetching Philadelphia...
Fetching Baltimore...
Fetching Newark...
Fetching Wilmington...
No. rows after cleaning: 119139


,PJME_MW,hour,dayofweek,month,quarter,year,dayofyear,dayofmonth,weekofyear,lag1,...,rolling_mean_24h,rolling_mean_168h,is_holiday,temperature,heating_degrees,cooling_degrees,rolling_temp_24h,rolling_temp_168h,is_weekend,is_peak_hour
Datetime,,,,,,,,,,,,,,,,,,,,,
2004-12-28 01:00:00,33781.0,1,1,12,4,2004,363,28,53,26506.0,...,36852.708333,34119.214286,0,-7.2,25.5,0.0,-4.263542,-0.352827,0,0


In [31]:
df.to_csv('../data/processed/PJME_hourly_processed.csv')